In [11]:
from assetextractor.extraction.utils import Config

from assetextractor.parsing.core.assets import Asset, AssetCache
from assetextractor.parsing.core.templates import Template
from assetextractor.parsing.core.properties import Property
from assetextractor.parsing.core.attributes import ListAttribute, GenericDictAttribute, FileNameAttribute, ListItem, ReferenceAttribute

import json
from pathlib import Path
from lxml.etree import tostring

import typing as t

In [2]:
config = Config.from_json("config.json")
assets = AssetCache.load(config)
templates = assets.templates

In [13]:
def find_filename_attributes(obj, paths=None):
    """Recursively find all FileNameAttribute instances and extract their paths."""
    if paths is None:
        paths = set()
    
    # Skip ReferenceAttribute to avoid infinite loops
    if isinstance(obj, ReferenceAttribute):
        return paths
    
    if isinstance(obj, FileNameAttribute) and obj.value:
        # Extract directory path from filename, excluding the filename itself
        file_path = Path(obj.value)
        if file_path.parent != Path('.'):  # Skip if no directory component
            paths.add(str(file_path.parent).replace('\\', '/'))
    
    # Recursively search in various container types
    if isinstance(obj, (ListAttribute, list)):
        for item in obj:
            find_filename_attributes(item, paths)
    elif isinstance(obj, (GenericDictAttribute, dict, Property, Asset)):
        for value in (obj.values() if isinstance(obj, dict) else obj):
            find_filename_attributes(value, paths)
    
    return paths

# Extract unique paths from all assets - using exact same iteration as convert.py
all_paths = set()

print("Extracting paths from assets...")
for asset in list(assets):
    find_filename_attributes(asset, all_paths)


# Sort paths for better readability
unique_paths = sorted(all_paths)

print(f"\\nFound {len(unique_paths)} unique directory paths:")
for path in unique_paths:
    print(f"  {path}")

Extracting paths from assets...
\nFound 520 unique directory paths:
  C:/dev/asset-extractor/.cache-117
  C:/dev/asset-extractor/.cache-117/data/base/config/portraitcam
  C:/dev/asset-extractor/.cache-117/data/base/graphics/celtic/buildings/ornaments/cultural
  C:/dev/asset-extractor/.cache-117/data/base/graphics/celtic/buildings/ornaments/grounds
  C:/dev/asset-extractor/.cache-117/data/base/graphics/celtic/buildings/ornaments/grounds/mosaic_plazas/mosaic_01
  C:/dev/asset-extractor/.cache-117/data/base/graphics/celtic/buildings/ornaments/grounds/mosaic_plazas/mosaic_02
  C:/dev/asset-extractor/.cache-117/data/base/graphics/celtic/buildings/ornaments/systems
  C:/dev/asset-extractor/.cache-117/data/base/graphics/celtic/buildings/ornaments/systems/colonnade_01
  C:/dev/asset-extractor/.cache-117/data/base/graphics/celtic/buildings/ornaments/systems/wall_01
  C:/dev/asset-extractor/.cache-117/data/base/graphics/celtic/buildings/production/field/barley
  C:/dev/asset-extractor/.cache-117